# DevFrogs Platform Knowledge — QLoRA fine-tune (free T4, Kaggle/Colab)

Fine-tunes **DeepSeek-R1-Distill-Qwen-1.5B** (swap to `-7B` once this works) with QLoRA using Unsloth so it
natively knows every DevFrogs division, app, and game — ports, tech stacks, features, auth model, the works.

**This notebook calls no external AI API — no Gemini, no OpenAI, nothing.** The only network calls are
downloading the base model weights from Hugging Face and, in cell 2, mounting your own Kaggle/Colab storage.
Training, the dataset, and inference are all local.

Designed to fit in a 16GB T4 and finish in well under an hour — the dataset is small (built from real
`STRUCTURE.md`/README facts, expanded with paraphrases) so this trains fast even on the free tier.

**Before running:**
1. On Kaggle: enable GPU (Settings → Accelerator → GPU T4 x2) and internet access.
2. On Colab: Runtime → Change runtime type → T4 GPU.
3. To teach it more (new apps, changed ports, new games), edit the `facts` list in the dataset cell —
   that's the only cell you need to touch as the platform evolves.
4. Checkpoints save every 25 steps — if your session dies, just re-run from the top; training auto-resumes
   from the last checkpoint in `/kaggle/working/checkpoints` (Kaggle) or
   `/content/drive/MyDrive/devfrogsai/checkpoints` (Colab).

In [1]:
# --- 1. Install deps (Unsloth handles the tricky bitsandbytes/xformers pinning for you) ---
!pip install -q -U unsloth
!pip install -q -U datasets trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 23.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 80.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 46.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 66.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 80.6 MB/s eta 0:00:00:0

In [2]:
# --- 2. Mount storage so checkpoints survive a session crash/disconnect ---
import os

ON_KAGGLE = os.path.exists("/kaggle/working")

if ON_KAGGLE:
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
    OUTPUT_DIR = "/kaggle/working/devfrogs-platform-lora"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/devfrogsai/checkpoints"
    OUTPUT_DIR = "/content/drive/MyDrive/devfrogsai/devfrogs-platform-lora"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Checkpoints ->", CHECKPOINT_DIR)
print("Final model ->", OUTPUT_DIR)

Checkpoints -> /kaggle/working/checkpoints
Final model -> /kaggle/working/devfrogs-platform-lora


In [3]:
# --- 3. Load base model in 4-bit (QLoRA) — fully local, no API key needed ---
from unsloth import FastLanguageModel
import torch

# Swap to "unsloth/DeepSeek-R1-Distill-Qwen-7B-bnb-4bit" once the 1.5B run works end to end.
MODEL_NAME = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect (bf16/fp16)
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: Will load unsloth/DeepSeek-R1-Distill-Qwen-1.5B-bnb-4bit as a legacy tokenizer.


In [4]:
# --- 4. Attach LoRA adapters (this is the only part that trains) ---
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                    # LoRA rank — 16 is a good default; try 8 if you OOM, 32 if you have headroom
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # cuts VRAM further — needed on a T4
    random_state=3407,
)
model.print_trainable_parameters()

Unsloth 2026.7.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 18,464,768 || all params: 1,795,552,768 || trainable%: 1.0284


In [5]:
# --- 5. Your dataset — real, repo-grounded DevFrogs platform + game knowledge ---
# Each `fact` has several paraphrased `questions` so the model learns the underlying fact,
# not just one exact question string. Add more facts/paraphrases here as the repo evolves;
# this is the ONLY thing you edit to grow the model's platform knowledge.

facts = [
  {
    "topic": "mission",
    "questions": [
      "What is DevFrogs' mission?",
      "Why does DevFrogs exist?",
      "Summarize what DevFrogs is trying to build."
    ],
    "answer": "DevFrogs' mission is to democratize AI — make rapid AI accessible to everyone — by building an integrated platform of foundational micro-apps and core utilities that is seamless, scalable, and practical from day one."
  },
  {
    "topic": "divisions",
    "questions": [
      "What divisions make up the DevFrogs monorepo?",
      "List DevFrogs' top-level divisions."
    ],
    "answer": "DevFrogs is organized into eight divisions: work (HRM, Jobs), studio (Office Suite, Projects, Plan2Fit), meet (Meet HD), learn (Education, ClassDesk, Timetable, Code, ipexam), play (Frogland, Shiftbound, Jobshift, PromptQuest), ai (Model Gateway, Shared AI, Agents, Research), lab (Prototypes, Incubation, Graduation), and core (Hub, Auth, Admin, Shared Auth UI)."
  },
  {
    "topic": "repo convention",
    "questions": [
      "What's the repo layout convention for a DevFrogs app?",
      "How is each app organized inside a division?"
    ],
    "answer": "Every app follows division/app/{frontend,backend[,app]} — e.g. work/jobs/backend, work/jobs/frontend, and sometimes work/jobs/app for a React Native client. The Docker build context is always the repo root, so Dockerfiles reference full paths like learn/classdesk/frontend/."
  },
  {
    "topic": "suites",
    "questions": [
      "What is the suites/ directory for?",
      "What are DevFrogs' commercial suites?"
    ],
    "answer": "suites/ defines DevFrogs' commercial pricing bundles — Project, Employee, and School suites — which package the underlying apps into sellable products rather than being apps themselves."
  },
  {
    "topic": "hrm overview",
    "questions": [
      "What is work/hrm?",
      "What does the HRM app do?"
    ],
    "answer": "work/hrm is DevFrogs' HR management app for 'Solene Labs' — employee records, org chart, onboarding, payroll, leave, recruiting, and performance reviews — with mobile and desktop layouts that switch automatically at a 900px breakpoint."
  },
  {
    "topic": "hrm stack",
    "questions": [
      "What tech stack and ports does work/hrm use?",
      "Which apps does work/hrm ship, and on what ports?"
    ],
    "answer": "work/hrm has an Express + Mongoose backend on port 4700, a React 19 + TypeScript + Vite + Tailwind web frontend on port 3700, and a React Native app (Android, iOS, macOS, Windows)."
  },
  {
    "topic": "hrm screens",
    "questions": [
      "What screens does work/hrm implement?",
      "List the main features of the HRM app."
    ],
    "answer": "work/hrm implements Inbox, Dashboard, Directory, Profile, Org Chart, Time Off, Onboarding, Recruiting, Payroll, and Performance screens, each with Admin/Manager/Employee role variants."
  },
  {
    "topic": "jobs overview",
    "questions": [
      "What is work/jobs?",
      "What does the Jobs app do?"
    ],
    "answer": "work/jobs is DevFrogs' job portal and ATS — job board, hiring pipeline, recruiter deep search, and verified skill profiles, implementing the JobPortal design prototype against a real Express + MongoDB API."
  },
  {
    "topic": "jobs stack",
    "questions": [
      "What tech stack and ports does work/jobs use?",
      "On what ports does the Jobs app run?"
    ],
    "answer": "work/jobs has an Express + Mongoose backend on port 4500, a React 19 + TypeScript + Vite web frontend on port 3900, and a React Native pilot app covering Board, My Applications, and Dashboard."
  },
  {
    "topic": "jobs screens",
    "questions": [
      "What screens does work/jobs implement?",
      "List the main features of the Jobs app."
    ],
    "answer": "work/jobs implements Board, My Applications, Company, Recruiter Search, Compare, Booking, Dashboard, Pipeline (kanban ATS), and Team screens."
  },
  {
    "topic": "jobs org model",
    "questions": [
      "How does organization/role management work in work/jobs?",
      "Who can create a hiring organization in the Jobs app?"
    ],
    "answer": "Every DevFrogs account is a 'candidate' by default in work/jobs. Only an account holding the platform-level 'org_admin' grant (assigned only from the DevFrogs Admin Console) can bootstrap a hiring organization and become its first in-app Admin; that Admin then assigns Admin/Hiring Manager/Interviewer roles to teammates from the in-app Team screen."
  },
  {
    "topic": "office overview",
    "questions": [
      "What is studio/office?",
      "What is OrbitStudio?"
    ],
    "answer": "studio/office is DevFrogs' Office Suite — Docs, Sheets, Slides, and Forms — internally also known as OrbitStudio (formerly unified/)."
  },
  {
    "topic": "projects overview",
    "questions": [
      "What is studio/projects?",
      "What does the Projects app do?"
    ],
    "answer": "studio/projects is a multi-tenant Employee & Project Management workspace — kanban board, people directory, analytics — with organizations able to define their own custom fields, status workflows, and roles."
  },
  {
    "topic": "projects stack",
    "questions": [
      "What tech stack and ports does studio/projects use?",
      "On what ports does the Projects app run?"
    ],
    "answer": "studio/projects has an Express + Mongoose + TypeScript backend on port 4550, a React 19 + Vite + TypeScript frontend on port 3450, and a React Native pilot app with Overview, Board, and People screens."
  },
  {
    "topic": "projects org model",
    "questions": [
      "How does organization/role management work in studio/projects?",
      "How are custom roles handled in the Projects app?"
    ],
    "answer": "studio/projects mirrors work/jobs' model: a platform-level 'org_admin' grant (from the Admin Console only) bootstraps one Organization, whose own admin then manages every other member's org-scoped role — including custom roles the org itself defines — from the in-app People screen."
  },
  {
    "topic": "plan2fit overview",
    "questions": [
      "What is studio/plan2fit?",
      "What does the Plan2Fit app do?"
    ],
    "answer": "studio/plan2fit is a consumer (B2C) fitness app: goal-based onboarding, workouts, bookings with QR check-in, progress tracking, and a workout library, built as a Vite + React + TypeScript frontend with an Express + MongoDB backend."
  },
  {
    "topic": "plan2fit roles",
    "questions": [
      "What user roles exist in studio/plan2fit?",
      "How is access controlled in Plan2Fit?"
    ],
    "answer": "studio/plan2fit uses JWT login/register with four roles: user, trainer, gym_admin, and superadmin."
  },
  {
    "topic": "meet overview",
    "questions": [
      "What is meet/meet-hd?",
      "What does the Meet HD app do?"
    ],
    "answer": "meet/meet-hd is DevFrogs' video app for live classes, webinars, and interview rooms (previously just meet/)."
  },
  {
    "topic": "education overview",
    "questions": [
      "What is learn/education?",
      "What does the Education app do?"
    ],
    "answer": "learn/education is DevFrogs' self-paced course platform — courses, video lessons, and certificates — organized as a feature-driven boilerplate where each backend module (courses, lessons, tests, results, progress) owns its own model, service, controller, and routes."
  },
  {
    "topic": "classdesk overview",
    "questions": [
      "What is learn/classdesk?",
      "What does the ClassDesk app do?"
    ],
    "answer": "learn/classdesk handles classrooms, cohorts, and assignments (previously classroom/)."
  },
  {
    "topic": "timetable overview",
    "questions": [
      "What is learn/timetable?",
      "What does the Timetable app do?"
    ],
    "answer": "learn/timetable is DevFrogs' TimeTable and AttendEase platform — class scheduling and attendance tracking — with its own web (Next.js), mobile (Expo/React Native), and Prisma-backed backend, incubating out of classdesk's frontend."
  },
  {
    "topic": "code overview",
    "questions": [
      "What is learn/code?",
      "What does the Code app do?"
    ],
    "answer": "learn/code handles coding challenges with sandboxed code execution (previously Code/)."
  },
  {
    "topic": "ipexam overview",
    "questions": [
      "What is learn/ipexam?",
      "Is learn/ipexam built yet?"
    ],
    "answer": "learn/ipexam is a planned app for proctored assessments with white-label support; it has not been built yet."
  },
  {
    "topic": "frogland overview",
    "questions": [
      "What is play/frogland?",
      "What kind of game is Frogland?"
    ],
    "answer": "play/frogland is a browser-first 3D survival game: a frog surviving a procedurally generated wetland, built with Three.js for rendering and bitECS for entity logic, designed to stay light on low-end hardware."
  },
  {
    "topic": "frogland tech",
    "questions": [
      "How does Frogland stay performant on weak hardware?",
      "What performance tricks does play/frogland use?"
    ],
    "answer": "Frogland streams only a 3x3 window of 16x16-tile chunks at a time, renders each chunk as one flat-shaded mesh, generates all textures at runtime on canvas (zero image assets), and saves only deltas to IndexedDB while regenerating terrain deterministically from a seed."
  },
  {
    "topic": "shiftbound overview",
    "questions": [
      "What is play/shiftbound?"
    ],
    "answer": "play/shiftbound is a games directory containing nested git repositories — shiftbound and CodeApp — with shiftbound originally scaffolded from a Google AI Studio export."
  },
  {
    "topic": "jobshift overview",
    "questions": [
      "What is play/jobshift?",
      "What does the Jobshift app do?"
    ],
    "answer": "play/jobshift is an umbrella hub for DevFrogs' skill-building career games — Draft, Architect, Lead, Patch, and PromptQuest — plus a Profile and Leaderboard, backed by a real Express + MongoDB API and real DevFrogs SSO (it superseded the earlier play/promptquest/)."
  },
  {
    "topic": "jobshift stack",
    "questions": [
      "What tech stack and ports does play/jobshift use?",
      "On what ports does Jobshift run?"
    ],
    "answer": "play/jobshift has an Express 5 + Mongoose backend on port 4800 and a React 19 + TypeScript + Vite frontend on port 3500."
  },
  {
    "topic": "jobshift auth",
    "questions": [
      "How does authentication work in play/jobshift?",
      "Does Jobshift have its own login?"
    ],
    "answer": "play/jobshift has no in-app login — every screen sits behind real DevFrogs SSO. The frontend uses useCentralAuth({appId: 'jobshift'}) to bounce unauthenticated visitors to auth.devfrogs.com, and the backend verifies the shared sso_token cookie against the hub."
  },
  {
    "topic": "jobshift games list",
    "questions": [
      "What games does play/jobshift include?",
      "List the five playable games in Jobshift."
    ],
    "answer": "play/jobshift includes five playable games: Draft (writing/communication), Architect (system design), Lead (people/delivery management), Patch (security triage), and PromptQuest (prompt engineering) — plus aggregate Profile and Leaderboard screens."
  },
  {
    "topic": "jobshift draft",
    "questions": [
      "What is the Draft game in Jobshift?",
      "How is Draft graded?"
    ],
    "answer": "Draft is a writing/communication game with tone-basics, structure, tables, visuals, multi-format, and capstone levels, graded on tone match, clarity, structure, and completeness."
  },
  {
    "topic": "jobshift architect",
    "questions": [
      "What is the Architect game in Jobshift?",
      "How is Architect graded?"
    ],
    "answer": "Architect is a component-palette system design game, graded on scalability, simplicity, cost-awareness, and correctness."
  },
  {
    "topic": "jobshift lead",
    "questions": [
      "What is the Lead game in Jobshift?",
      "How is Lead graded?"
    ],
    "answer": "Lead is a people-management game with move and message levels plus a periodic live call/meeting bonus round using browser speech-to-text, graded on delivery, budget, people, and communication."
  },
  {
    "topic": "jobshift patch",
    "questions": [
      "What is the Patch game in Jobshift?",
      "How is Patch graded?"
    ],
    "answer": "Patch is a security triage game over vulnerable code, with a toggle between fixing code directly or prompting an AI to fix it, graded on vulnerability identification, fix quality, prompt/response quality, and judgment."
  },
  {
    "topic": "jobshift promptquest",
    "questions": [
      "What is the PromptQuest game in Jobshift?",
      "How is PromptQuest graded?"
    ],
    "answer": "PromptQuest is a prompt-engineering game where players rewrite broken prompts, graded on clarity, specificity, context, and constraints."
  },
  {
    "topic": "jobshift profile leaderboard",
    "questions": [
      "What do the Profile and Leaderboard screens in Jobshift show?"
    ],
    "answer": "Jobshift's Profile screen aggregates XP/level across all five games with per-game progress bars and recent submission activity; the Leaderboard ranks players by total XP across every game."
  },
  {
    "topic": "promptquest status",
    "questions": [
      "What is play/promptquest now?",
      "Is play/promptquest still the active game hub?"
    ],
    "answer": "play/promptquest has been superseded by play/jobshift and is kept only as a pointer to it."
  },
  {
    "topic": "model gateway overview",
    "questions": [
      "What is ai/model-gateway?"
    ],
    "answer": "ai/model-gateway hosts RAG and flow workflows (previously flow/) — it's the gateway layer meant to grow with more AI capability features over time."
  },
  {
    "topic": "shared ai overview",
    "questions": [
      "What is ai/shared-ai?",
      "What does the @devfrogs/ai-service package provide?"
    ],
    "answer": "ai/shared-ai is the @devfrogs/ai-service package — shared Gemini utilities used across DevFrogs backends, including a Gemini client, an SSE streaming helper, a per-user sliding-window rate limiter, and context-building helpers for documents and user profiles."
  },
  {
    "topic": "agents overview",
    "questions": [
      "What is ai/agents?",
      "What copilots are planned for ai/agents?"
    ],
    "answer": "ai/agents is a planned home for DevFrogs' copilots — an AI tutor for Learn, code hints for Code, resume/JD matching for Jobs, a doc copilot for Studio, meeting summaries for Meet, and exam generation for ipexam — not built yet."
  },
  {
    "topic": "research overview",
    "questions": [
      "What is ai/research?"
    ],
    "answer": "ai/research is DevFrogs' ML research area for custom models, fine-tunes, and papers."
  },
  {
    "topic": "lab overview",
    "questions": [
      "What is the lab/ division for?",
      "What are lab/prototypes, lab/incubation, and lab/graduation?"
    ],
    "answer": "lab/ is DevFrogs' R&D division: lab/prototypes holds 4-6 week timeboxed spikes, lab/incubation validates ideas with real users, and lab/graduation is the gate deciding who pays for an idea and which division should own it going forward."
  },
  {
    "topic": "hub overview",
    "questions": [
      "What is core/hub?",
      "What does the Hub do?"
    ],
    "answer": "core/hub is DevFrogs' SSO and OIDC provider (previously main_df/), split into backend_main_df (the central auth API, sessions, 'Sign in with DevFrogs') and frontend_main_df (the devfrogs.com landing page, dashboard, and /developers portal); its backend defaults to port 4000."
  },
  {
    "topic": "auth overview",
    "questions": [
      "What is core/auth?"
    ],
    "answer": "core/auth is DevFrogs' central login page, served at auth.devfrogs.com."
  },
  {
    "topic": "admin overview",
    "questions": [
      "What is core/admin?",
      "What does the Admin Console manage?"
    ],
    "answer": "core/admin is the DevFrogs Admin Console — managing tenants, per-app roles (like the 'org_admin' grants used by Jobs and Projects), and plan toggles — secured with its own JWT and MFA."
  },
  {
    "topic": "shared auth ui overview",
    "questions": [
      "What is core/shared-auth-ui?"
    ],
    "answer": "core/shared-auth-ui is the @devfrogs/auth-ui package — the shared login UI and central-auth helpers (like useCentralAuth) that every frontend imports for 'Sign in with DevFrogs'."
  },
  {
    "topic": "sso pattern",
    "questions": [
      "How does 'Sign in with DevFrogs' work across apps?",
      "What's the shared authentication pattern across DevFrogs apps?"
    ],
    "answer": "Apps like work/jobs, work/hrm, studio/projects, and play/jobshift have no in-app login form — they bounce unauthenticated visitors to the central login page (auth.devfrogs.com) via @devfrogs/auth-ui, and their backends verify the shared sso_token cookie against core/hub's /api/auth/me endpoint."
  },
  {
    "topic": "react native apps",
    "questions": [
      "Which DevFrogs apps have a React Native client?"
    ],
    "answer": "work/hrm, work/jobs, and studio/projects each have a React Native app alongside their web frontend and backend — hrm's covers Android, iOS, macOS, and Windows, while jobs and projects ship smaller pilot slices."
  },
  {
    "topic": "mongo fallback",
    "questions": [
      "What happens if MongoDB isn't reachable for a DevFrogs backend?"
    ],
    "answer": "Backends like work/hrm, work/jobs, studio/projects, and play/jobshift automatically fall back to an in-memory MongoDB instance for local development when their configured MONGODB_URI isn't reachable — data just resets on restart in that mode."
  },
  {
    "topic": "games list",
    "questions": [
      "What games exist across the whole play/ division?",
      "List every game in DevFrogs' play division."
    ],
    "answer": "play/ contains Frogland (a 3D survival game), Shiftbound and CodeApp (nested game repos), and Jobshift's five career-skill games — Draft, Architect, Lead, Patch, and PromptQuest — with play/promptquest kept only as a pointer to Jobshift."
  }
]

platform_data = []
for fact in facts:
    for q in fact["questions"]:
        platform_data.append({"instruction": q, "input": "", "output": fact["answer"]})

print(f"{len(facts)} facts -> {len(platform_data)} training examples (with paraphrases)")

PROMPT_TEMPLATE = """### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

def format_example(ex):
    return {
        "text": PROMPT_TEMPLATE.format(
            instruction=ex["instruction"], input=ex.get("input", ""), output=ex["output"]
        ) + tokenizer.eos_token
    }

from datasets import Dataset
dataset = Dataset.from_list(platform_data).shuffle(seed=3407).map(format_example)
print(dataset[0]["text"])

50 facts -> 93 training examples (with paraphrases)


Map:   0%|          | 0/93 [00:00<?, ? examples/s]

### Instruction:
On what ports does Jobshift run?

### Input:


### Response:
play/jobshift has an Express 5 + Mongoose backend on port 4800 and a React 19 + TypeScript + Vite frontend on port 3500.<｜end▁of▁sentence｜>


In [ ]:
# --- 6. Trainer — small repetitive dataset, so a few epochs over it is enough to converge ---
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=TrainingArguments(
        output_dir=CHECKPOINT_DIR,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,   # effective batch size 8
        num_train_epochs=5,              # small dataset — needs a few passes to stick
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        save_steps=25,
        save_total_limit=3,              # keep disk usage bounded
        optim="adamw_8bit",              # lower memory optimizer, matters on a T4
        seed=3407,
        report_to="none",
    ),
)

# Resume automatically if a checkpoint already exists from a previous session
import glob
existing_checkpoints = glob.glob(f"{CHECKPOINT_DIR}/checkpoint-*")
resume_from = max(existing_checkpoints, key=os.path.getmtime) if existing_checkpoints else None
if resume_from:
    print("Resuming from", resume_from)

trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# --- 7. Save the LoRA adapter (small — tens of MB, easy to move around) ---
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to", OUTPUT_DIR)

In [ ]:
# --- 8. Sanity check — ask it real platform questions, no API involved, all local inference ---
FastLanguageModel.for_inference(model)

test_questions = [
    "What port does the studio/projects backend run on?",
    "What games does play/jobshift include?",
    "What is Frogland built with?",
    "How does organization/role management work in work/jobs?",
]

for q in test_questions:
    prompt = PROMPT_TEMPLATE.format(instruction=q, input="", output="")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=120, use_cache=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(response.split("### Response:")[-1].strip())
    print("-" * 60)

## Next steps
- Grow `facts` in the dataset cell as the repo evolves — add new apps, correct ports that changed,
  add more paraphrases for anything the sanity check got wrong. Re-run from cell 5 down.
- Once this 1.5B run works cleanly end-to-end, change `MODEL_NAME` to
  `unsloth/DeepSeek-R1-Distill-Qwen-7B-bnb-4bit` and rerun — same code, same T4, just slower and lower
  `per_device_train_batch_size` (try 1-2) if you hit an OOM.
- To merge the LoRA adapter into a standalone model for fully local serving (still no API, no Gemini):
  `model.save_pretrained_merged(OUTPUT_DIR + "-merged", tokenizer, save_method="merged_16bit")`, then
  optionally convert to GGUF and run it with `ollama create devfrogs-platform -f Modelfile` for a local
  chat server on your own machine.
- To resume across a fresh Kaggle/Colab session on a different day: re-run cells 1–4, then just re-run
  cell 6 — it picks up the latest checkpoint automatically as long as `CHECKPOINT_DIR` points at the same
  persistent storage.
- **Caveat worth knowing:** this dataset teaches ~50 facts with a few paraphrases each — great for a model
  that natively "speaks DevFrogs," but it will go stale the moment the repo changes, and it won't generalize
  to questions far outside these facts. For something that always reflects the live repo, pair this with (or
  replace it with) retrieval over the actual docs (`ai/model-gateway`'s RAG pipeline) instead of relying only
  on baked-in weights.